# 02 · Dựng giao dịch Fashion-MNIST (TC1–TC7)

**Mục đích:** tạo bộ giao dịch S0 cho đủ 70.000 ảnh với đúng một codebook B0 / K=32 / seed=123.

**Điều kiện chạy:** Notebook 01 đã tạo dataset manifest trong cùng lượt; Julia 1.13 và môi trường notebook đã cài dependencies.

**Đầu vào:** dataset manifest, bốn archive đã xác minh và profile `src/applications/Fashion-Mnist/configs/transactions/release.toml`.

**Đầu ra:** một codebook, patch assignments, `transactions.jsonl`, ánh xạ TID–ảnh, metadata nhãn và `transactions/tc7_manifest.toml` dưới run root.


## 1. Nạp môi trường và tìm dataset manifest

Cell này tìm run manifest duy nhất hoặc dùng `FASHION_RELEASE_RUN_ID` nếu có nhiều lượt. Nó khởi tạo profile release và kiểm tra hash archive trước khi TC bắt đầu.


In [ ]:
using Pkg
using TOML

function find_repo_root(start=pwd())
    current = realpath(start)
    while true
        if isfile(joinpath(current, "Project.toml")) && isfile(joinpath(current, "src", "EclatFIM.jl"))
            return current
        end
        parent = dirname(current)
        parent == current && error("Không tìm thấy repository root")
        current = parent
    end
end

function find_release_manifest(repo_root)
    release_root = joinpath(repo_root, "results", "fashion", "release")
    isdir(release_root) || error("Chạy Notebook 01 trước để tạo results/fashion/release/<run_id>")
    candidates = [joinpath(path, "dataset", "dataset_manifest.toml")
        for path in readdir(release_root; join=true)
        if isfile(joinpath(path, "dataset", "dataset_manifest.toml"))]
    requested = get(ENV, "FASHION_RELEASE_RUN_ID", "")
    if !isempty(requested)
        selected = joinpath(release_root, requested, "dataset", "dataset_manifest.toml")
        isfile(selected) || error("Không có dataset manifest cho run ID $requested")
        return selected
    end
    length(candidates) == 1 || error("Có $(length(candidates)) run manifests; đặt FASHION_RELEASE_RUN_ID để chọn rõ một lượt")
    return only(candidates)
end

repo_root = get(ENV, "FASHION_MNIST_REPO_ROOT", find_repo_root())
Pkg.activate(repo_root)
using EclatFIM
TC = EclatFIM.FashionTC
manifest_path = find_release_manifest(repo_root)
dataset_manifest = EclatFIM.read_dataset_manifest(manifest_path;
    verify_archives=true, repo_root)
run_root = joinpath(repo_root, dataset_manifest["run_root"])
raw_data_dir = joinpath(repo_root, "data", "raw", "fashion-mnist")
config_path = joinpath(EclatFIM.FASHION_MNIST_ROOT, "configs", "transactions", "release.toml")
println("Run ID: ", dataset_manifest["run_id"])
println("Dataset manifest: ", relpath(manifest_path, repo_root))
println("Profile: ", relpath(config_path, repo_root))


## 2. Tạo trạng thái chạy và chạy smoke checks

Cell này nạp đủ train/test từ manifest, khóa mode `release` vào B0/K32/seed123/S0 và chạy các assertion toy của TC. Kiểm tra số ảnh là 70.000, stage bắt đầu từ 0 và mọi smoke check đạt.


In [2]:
RUN_STATE = TC.prepare_run(config_path; repo_root, data_dir=raw_data_dir,
    dataset_manifest_path=manifest_path)
SMOKE_CHECKS = TC.run_unit_smoke_checks!()
@assert length(RUN_STATE.image_ids) == 70_000
@assert all(==("PASS"), values(SMOKE_CHECKS))
println("Loaded ", length(RUN_STATE.image_ids), " images; four archive hashes verified.")
SMOKE_CHECKS


Loaded 70000 images; four archive hashes verified.


Dict{String, String} with 6 entries:
  "jsonl_empty_transaction_roundtrip"  => "PASS"
  "deterministic_tie_break"            => "PASS"
  "method_selector_and_invalid_value"  => "PASS"
  "transaction_sort_deduplicate_empty" => "PASS"
  "threshold_29_30"                    => "PASS"
  "column_major_descriptor"            => "PASS"

## 3. TC1 — Chia mỗi ảnh thành 49 patch 4×4

Cell này trích patch không overlap và giữ `patch_id`, hàng, cột. Kiểm tra tổng số patch trước lọc là 70.000 × 49 và các tọa độ nằm trên lưới 7×7.


In [3]:
TC1_RESULT = TC.run_tc1!(RUN_STATE)
@assert TC1_RESULT.patches_before_filter == 70_000 * 49
TC1_RESULT

(images = 70000, patches_before_filter = 3430000, patch_size = (4, 4), traversal = "row_major")

## 4. TC2 — Lọc patch foreground với ngưỡng `pixel > 29`

Cell này loại patch chỉ có pixel ≤29; ảnh không còn patch vẫn được giữ lại đến TC6. Kiểm tra số patch được giữ và số transaction rỗng được báo cáo.


In [4]:
TC2_RESULT = TC.run_tc2!(RUN_STATE)
TC2_RESULT


(threshold = 29, kept = 2143746, removed = 1286254, total_patches = 3430000, retention_rate = 0.6249988338192419, images_with_no_patches = 0)

## 5. TC3 — Tạo descriptor raw pixel 16 chiều

Cell này chuyển từng patch giữ lại thành vector `Float32`, chia 255 và giữ mapping ảnh/patch. Kết quả mong đợi có 16 hàng, một cột trên mỗi patch được giữ.


In [5]:
TC3_RESULT = TC.run_tc3!(RUN_STATE)
@assert size(RUN_STATE.descriptor_matrix, 1) == 16
TC3_RESULT


(descriptor_shape = (16, 2143746), retained_descriptors = 2143746, dtype = "Float32", flatten_order = "column_major")

## 6. TC4 — Fit duy nhất codebook B0/K=32/seed=123

Cell này chạy full-batch K-means trên descriptor từ toàn bộ 70.000 ảnh, không lấy mẫu ảnh hay descriptor. Kiểm tra notebook chỉ tạo đúng một codebook với cấu hình đã chốt.


In [6]:
TC4_RESULT = TC.run_tc4!(RUN_STATE)
@assert TC4_RESULT.codebooks == 1
@assert TC4_RESULT.run_ids == ["FashionMNIST_T2_S0_B0_K32_seed123"]
TC4_RESULT


[TC4] START method=raw_full_batch_kmeans | fits=1 | K=32 | seeds=123 | descriptors=2143746 | backend=Clustering.jl CPU
[TC4 1/1] START FashionMNIST_T2_S0_B0_K32_seed123 | descriptors=2143746 | maxiter=300 | tol=0.0001
[TC4 1/1] FITTED FashionMNIST_T2_S0_B0_K32_seed123 | iterations=139 | converged=true | distortion=0.347294 | occupancy=32/32 | top_share=0.0974 | entropy=0.9355 | fit=58.51s | checkpoint=/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/release/fashion_47265722452561/transactions/checkpoint.toml


(method = "raw_full_batch_kmeans", codebooks = 1, run_ids = ["FashionMNIST_T2_S0_B0_K32_seed123"], descriptors = 2143746, status_counts = Dict("FITTED" => 1, "DEGENERATE_WARNING" => 0, "NON_CONVERGED" => 0))

## 7. TC5 — Gán mỗi patch vào centroid gần nhất

Cell này thực hiện hard assignment theo squared Euclidean và lưu patch assignments cùng tọa độ. Kiểm tra số assignment bằng số descriptor TC3.


In [7]:
TC5_RESULT = TC.run_tc5!(RUN_STATE)
@assert only(TC5_RESULT).assignments == size(RUN_STATE.descriptor_matrix, 2)
TC5_RESULT


1-element Vector{NamedTuple}:
 (run_id = "FashionMNIST_T2_S0_B0_K32_seed123", assignments = 2143746, k = 32)

## 8. TC6 — Dựng transaction S0

Cell này lấy visual word ID duy nhất, sắp xếp tăng dần cho từng ảnh và giữ cả transaction rỗng. Nhãn chỉ nằm ở metadata; kiểm tra số transaction bằng 70.000.


In [8]:
TC6_RESULT = TC.run_tc6!(RUN_STATE)
@assert only(TC6_RESULT).transactions == 70_000
TC6_RESULT


1-element Vector{NamedTuple}:
 (run_id = "FashionMNIST_T2_S0_B0_K32_seed123", transactions = 70000, empty = 0)

## 9. TC7 — Đọc lại artifact và ghi manifest bàn giao

Cell này đọc lại codebook, assignments, transactions và TID mapping; sau đó ghi manifest có hash từng artifact cho Notebook 03/04. Kiểm tra `PASS`, một codebook, đúng 70.000 dòng và hash transaction.


In [9]:
TC7_RESULT = TC.run_tc7!(RUN_STATE)
@assert RUN_STATE.stage == 7
@assert length(TC7_RESULT.runs) == 1
@assert only(TC7_RESULT.runs).transactions == 70_000
@assert isfile(joinpath(run_root, "transactions", "tc7_manifest.toml"))
TC7_RESULT


(summary_path = "/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/release/fashion_47265722452561/tc_v2_run_summary.toml", runs = NamedTuple[(run_id = "FashionMNIST_T2_S0_B0_K32_seed123", status = "COMPLETED", fit_status = "FITTED", transactions = 70000, empty = 0, support_denominator = 70000, manifest_path = "/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/release/fashion_47265722452561/transactions/tc7_manifest.toml")], output_root = "/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/release/fashion_47265722452561", elapsed_seconds = 72.53918790817261)

## Kết quả lượt chạy

Các artifact nằm trong `results/fashion/release/<run_id>/transactions/`. Notebook 03 đọc `tc7_manifest.toml`; manifest này khóa transaction hash, mapping ảnh, assignments, codebook và cấu hình.


In [10]:
println("TC stages: ", RUN_STATE.stage, "/7")
println("TC7 manifest: ", relpath(joinpath(run_root, "transactions", "tc7_manifest.toml"), repo_root))


TC stages: 7/7
TC7 manifest: results/fashion/release/fashion_47265722452561/transactions/tc7_manifest.toml
